In [ ]:
## pull the data with links
# --- Initialize data accumulation storage BEFORE the loop ---
all_table_data = []
headers = []  # We will capture headers on the first successful pass

# -----------------------------
# 1. Get the total number of options first safely
# -----------------------------
state_dropdown = wait.until(
    EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='state']"))
)
total_options = len(Select(state_dropdown).options)
print(total_options)
# -----------------------------
# 2. Loop by index
# -----------------------------
for index in range(1, total_options):  # Start at 1 to skip placeholder
    #if index == 1:
        #continue
    state_dropdown = wait.until(
        EC.visibility_of_element_located((By.XPATH, "//select[@formcontrolname='state']"))
    )
    state_select = Select(state_dropdown)
    state_value = state_select.options[index].get_attribute("value")
    
    state_select.select_by_index(index)
    print(f"\nProcessing state index {index} (Value: {state_value})...")

    # Click Search Button Safely
    search_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[@type='submit' and contains(.,'Search')]"))
    )
    
    existing_tables = driver.find_elements(By.ID, "excel-table")
    old_table = existing_tables[0] if existing_tables else None

    driver.execute_script("arguments[0].click();", search_button)
    print(f"Search successfully forced click for: {state_value}")
    
    if old_table:
        try:
            wait.until(EC.staleness_of(old_table))
        except Exception:
            time.sleep(1) 

    # Handle Missing Table if No Results Exist
    try:
        wait.until(EC.visibility_of_element_located((By.ID, "excel-table")))
        print(f"Table successfully loaded for state: {state_value}")
    except TimeoutException:
        print(f"⚠️ No results found (Timeout) for state: {state_value}. Skipping...")
        time.sleep(1)
        continue

    # -----------------------------
    # LIVE ROW CLICKING & SCRAPING LOGIC 
    # -----------------------------
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find("table", {"id": "excel-table"})
    
    if table:
        tbody = table.find("tbody")
        if tbody:
            rows_bs = tbody.find_all("tr")
            
            tbody_text = tbody.get_text(strip=True).lower()
            if "no record" in tbody_text or "no data" in tbody_text or not rows_bs:
                print(f"ℹ️ Table contains explicit empty notice for state: {state_value}. Skipping...")
                time.sleep(1)
                continue

            if not headers:
                thead = table.find("thead")
                if thead:
                    for th in thead.find_all("th"):
                        headers.append(th.get_text(strip=True))

            total_rows = len(rows_bs)
            print(f"Found {total_rows} proposals to process for state {state_value}.")

            # Loop through rows sequentially
            for row_idx in range(1, total_rows + 1):
                try:
                    cols_elements = driver.find_elements(By.XPATH, f"//table[@id='excel-table']/tbody/tr[{row_idx}]/td")
                    row_data = [col.text.strip() for col in cols_elements]
                    
                    if not row_data:
                        continue
                    
                    proposal_link = driver.find_element(By.XPATH, f"//table[@id='excel-table']/tbody/tr[{row_idx}]/td[2]/a")
                    proposal_no = proposal_link.text.strip()
                    
                    print(f" -> Clicking proposal {row_idx}/{total_rows}: {proposal_no}")
                    
                    main_window = driver.current_window_handle
                    driver.execute_script("arguments[0].click();", proposal_link)
                    time.sleep(3)  
                    
                    details_url = "N/A"
                    view_proposal_url = "N/A"
                    
                    opened_in_new_tab = len(driver.window_handles) > 1
                    if opened_in_new_tab:
                        details_window = [w for w in driver.window_handles if w != main_window][0]
                        driver.switch_to.window(details_window)
                    
                    details_url = driver.current_url
                    
                    # -------------------------------------------------------------
                    # NESTED EXTRACTION: Click "View Proposal" & Capture Document URL
                    # -------------------------------------------------------------
                    try:
                        view_proposal_element = driver.find_element(By.XPATH, "//a[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'view proposal')]")
                        pre_click_windows = driver.window_handles
                        
                        driver.execute_script("arguments[0].click();", view_proposal_element)
                        time.sleep(3)
                        
                        post_click_windows = driver.window_handles
                        if len(post_click_windows) > len(pre_click_windows):
                            proposal_doc_window = [w for w in post_click_windows if w not in pre_click_windows][0]
                            driver.switch_to.window(proposal_doc_window)
                            
                            view_proposal_url = driver.current_url  
                            
                            driver.close()  # Close document tab
                            if opened_in_new_tab:
                                driver.switch_to.window(details_window)
                            else:
                                driver.switch_to.window(main_window)
                        else:
                            view_proposal_url = driver.current_url
                            driver.back()
                            time.sleep(1)
                            
                    except NoSuchElementException:
                        print("    ⚠️ 'View Proposal' link/button was not found on this view layout.")
                    except Exception as nested_err:
                        print(f"    ❌ Failed tracking 'View Proposal' sub-route: {nested_err}")
                    
                    # -------------------------------------------------------------
                    # BACKWARDS ROUTING CLEANUP & RE-ALIGNMENT
                    # -------------------------------------------------------------
                    if opened_in_new_tab:
                        driver.close() 
                        driver.switch_to.window(main_window)
                    else:
                        driver.back() 
                        wait.until(EC.visibility_of_element_located((By.ID, "excel-table")))
                    
                    # Compile target URL metrics back into data array rows
                    row_data.append(details_url)        
                    row_data.append(view_proposal_url)   
                    row_data.append(state_value)        
                    all_table_data.append(row_data)
                    print(f"    Successfully Scraped URLs.")
                    
                except NoSuchElementException:
                    print(f" ⚠️ Could not find target link anchor for index row {row_idx}. Skipping...")
                    continue
                except Exception as e:
                    print(f" ❌ Error processing index row {row_idx}: {e}")
                    all_windows = driver.window_handles
                    if len(all_windows) > 1:
                        for extra_w in all_windows[1:]:
                            driver.switch_to.window(extra_w)
                            driver.close()
                    driver.switch_to.window(main_window)
                    continue

    time.sleep(1)
    #break  # Kept active for test validation. Remove or comment out to run for all states.

# -----------------------------
# 3. Post-Loop Data Compilation
# -----------------------------
if all_table_data:
    expected_header_count = len(all_table_data[0])
    
    if len(headers) < expected_header_count:
        headers.append("Details_URL")      
        headers.append("Proposal_URL")     
        headers.append("State_Value")
        
    df = pd.DataFrame(all_table_data, columns=headers)
    print("\n--- Final Extracted Dataset Preview ---")
    print(df.head()) 

    file_path = "parivesh_data.csv"

    # Check if the file already exists
    file_exists = os.path.exists(file_path)

    # Write to CSV
    df.to_csv(
    file_path, 
    mode='a', 
    index=False, 
    header=not file_exists  # Writes header ONLY if the file does NOT exist yet
)
    print("\nAll available states scraped and saved to parivesh_data.csv successfully!")
else:
    print("\n❌ Automation complete. Zero data entries found across all processed states.")